# Keep context during one investigation

This notebook gives one AgentScope agent a temporary conversation memory. The agent will use a detail from the first message when it answers a later message.

## What short-term memory means here

In the current AgentScope 2 API, the agent's `AgentState.context` list stores messages while this notebook's agent object exists. It is useful for following a conversation, but it is not a permanent case record and it does not verify evidence.

In [ ]:
import os

from dotenv import load_dotenv

from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
# AgentState holds the current conversation context and other temporary agent state.
from agentscope.state import AgentState
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel

## Configure the local model

The next cell reads the local model settings from `.env` and creates the model connection that the case agent will use.

In [ ]:
# Read the local-model settings from .env rather than hard-coding them.
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    # Consistent, short responses make the memory behavior easier to inspect.
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=180),
)

## Create the memory and the agent

The `case_state` variable is separate from the agent so that we can inspect and clear it explicitly. Passing it as `state=` gives the agent an explicit conversation state. Its `context` list holds the earlier messages used on later turns.

In [ ]:
# The state starts with an empty context. It will grow as the agent receives and sends messages.
case_state = AgentState()

case_agent = Agent(
    name="case_assistant",
    system_prompt=(
        "You help summarize a practice security case. "
        "Use earlier conversation details when they are available. "
        "Do not invent facts that were not provided."
    ),
    model=model,
    # This is the important new argument in Lesson 05.
    state=case_state,
    react_config=ReActConfig(max_iters=3),
)

print(f"Created {case_agent.name} with an empty short-term context.")

## Give the agent the first case detail

The next cell sends the incident number and preliminary assessment. AgentScope stores this user message and the reply in the agent state for a later turn.

In [ ]:
# First turn: provide the facts the agent should remember during this case.
first_question = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text=(
        "For practice case INC-204, the preliminary assessment is 'needs review'. "
        "Reply with a one-sentence acknowledgement."
    ))],
)

first_response = await case_agent.reply(first_question)
print("".join(block.text for block in first_response.content if isinstance(block, TextBlock)))

## Ask a follow-up that needs earlier context

The next cell does not repeat the case details. The response should show that the agent used the earlier messages retained in `case_state.context`.

In [ ]:
# Second turn: do not repeat the incident number or assessment. The agent
# should find them in case_state.context before composing its response.
follow_up = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text=(
        "What is the incident number and its preliminary assessment? "
        "Answer in one sentence."
    ))],
)

follow_up_response = await case_agent.reply(follow_up)
print("".join(block.text for block in follow_up_response.content if isinstance(block, TextBlock)))

## Inspect the stored conversation

The next cell prints the messages that AgentScope retained in `case_state.context`. This makes the state visible: the model does not receive a mysterious memory; it receives earlier conversation context stored by the agent.

In [ ]:
# AgentState.context is the list of messages currently available to the agent.
stored_messages = case_state.context
print(f"Stored messages: {len(stored_messages)}")

for number, message in enumerate(stored_messages, start=1):
    # get_text_content() provides a readable version of each stored message.
    print(f"{number}. {message.role} ({message.name}): {message.get_text_content()}")

## Clear the case context

Clearing short-term memory prevents an old practice case from influencing an unrelated one. Run this cell after inspecting the conversation.

In [ ]:
# This deletes only the temporary context in case_state.
# It does not modify files, external services, or any permanent case record.
case_state.context.clear()
case_state.summary = ""

remaining_messages = case_state.context
print(f"Stored messages after clearing: {len(remaining_messages)}")

## Checkpoint

Change `INC-204` to `INC-319` and `needs review` to `benign` in the first-turn cell. Rerun the agent-creation, first-turn, and follow-up-turn cells. The follow-up should use the new details. Then clear the memory and confirm that the count is zero.